<a href="https://colab.research.google.com/github/Madankk-06/Generative_AI_Projects/blob/main/2_Custom_Byte_Pair_Encoding_(BPE)_Tokenizer_%26_Autoregressive_Causal_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
from collections import Counter, defaultdict

np.random.seed(42)

In [2]:
corpus = [
    "Artificial intelligence is transforming the world.",
    "Machine learning enables computers to learn from data.",
    "Deep learning uses neural networks.",
    "Transformers are powerful language models.",
    "Byte Pair Encoding builds subword vocabularies.",
    "Attention mechanisms improve sequence modeling.",
    "Language models generate meaningful text.",
    "Natural language processing powers chatbots.",
    "PyTorch provides efficient tensor operations.",
    "Custom tokenizers improve flexibility."
]

print("Number of sentences:", len(corpus))

for sentence in corpus:
    print(sentence)

Number of sentences: 10
Artificial intelligence is transforming the world.
Machine learning enables computers to learn from data.
Deep learning uses neural networks.
Transformers are powerful language models.
Byte Pair Encoding builds subword vocabularies.
Attention mechanisms improve sequence modeling.
Language models generate meaningful text.
Natural language processing powers chatbots.
PyTorch provides efficient tensor operations.
Custom tokenizers improve flexibility.


In [3]:
def preprocess(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^a-z ]', '', sentence)
    return sentence

processed = [preprocess(s) for s in corpus]

print(processed)

['artificial intelligence is transforming the world', 'machine learning enables computers to learn from data', 'deep learning uses neural networks', 'transformers are powerful language models', 'byte pair encoding builds subword vocabularies', 'attention mechanisms improve sequence modeling', 'language models generate meaningful text', 'natural language processing powers chatbots', 'pytorch provides efficient tensor operations', 'custom tokenizers improve flexibility']


In [4]:
vocab = Counter()

for sentence in processed:
    words = sentence.split()

    for word in words:
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] += 1

print("Vocabulary Size:", len(vocab))

print("\nSample Entries:\n")

for i, (word, freq) in enumerate(vocab.items()):
    print(word, ":", freq)

    if i == 8:
        break

Vocabulary Size: 49

Sample Entries:

a r t i f i c i a l </w> : 1
i n t e l l i g e n c e </w> : 1
i s </w> : 1
t r a n s f o r m i n g </w> : 1
t h e </w> : 1
w o r l d </w> : 1
m a c h i n e </w> : 1
l e a r n i n g </w> : 2
e n a b l e s </w> : 1


In [5]:
def get_pair_frequencies(vocab):

    pairs = defaultdict(int)

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])

            pairs[pair] += freq

    return pairs

pairs = get_pair_frequencies(vocab)

print("Number of Unique Pairs:", len(pairs))

print("\nTop 10 Token Pairs:\n")

for pair, count in sorted(
        pairs.items(),
        key=lambda x: x[1],
        reverse=True)[:10]:

    print(pair, "->", count)

Number of Unique Pairs: 166

Top 10 Token Pairs:

('s', '</w>') -> 16
('e', '</w>') -> 12
('n', 'g') -> 10
('i', 'n') -> 9
('e', 'n') -> 9
('t', 'e') -> 7
('a', 'n') -> 7
('o', 'r') -> 7
('e', 'r') -> 7
('a', 'r') -> 6


In [6]:
def get_pair_frequencies(vocab):

    pairs = defaultdict(int)

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])

            pairs[pair] += freq

    return pairs

pairs = get_pair_frequencies(vocab)

print("Number of Unique Pairs:", len(pairs))

print("\nTop 10 Token Pairs:\n")

for pair, count in sorted(
        pairs.items(),
        key=lambda x: x[1],
        reverse=True)[:10]:

    print(pair, "->", count)

Number of Unique Pairs: 166

Top 10 Token Pairs:

('s', '</w>') -> 16
('e', '</w>') -> 12
('n', 'g') -> 10
('i', 'n') -> 9
('e', 'n') -> 9
('t', 'e') -> 7
('a', 'n') -> 7
('o', 'r') -> 7
('e', 'r') -> 7
('a', 'r') -> 6


In [7]:
def merge_vocab(pair, vocab):

    merged_vocab = {}

    bigram = " ".join(pair)

    replacement = "".join(pair)

    for word in vocab:

        new_word = word.replace(bigram, replacement)

        merged_vocab[new_word] = vocab[word]

    return merged_vocab

In [8]:
num_merges = 30

bpe_vocab = vocab.copy()

merges = []

for i in range(num_merges):

    pairs = get_pair_frequencies(bpe_vocab)

    if not pairs:
        break

    best = max(pairs, key=pairs.get)

    merges.append(best)

    bpe_vocab = merge_vocab(best, bpe_vocab)

    print(f"Merge {i+1:02d}: {best}")

Merge 01: ('s', '</w>')
Merge 02: ('e', '</w>')
Merge 03: ('n', 'g')
Merge 04: ('e', 'n')
Merge 05: ('o', 'r')
Merge 06: ('i', 'ng')
Merge 07: ('e', 'r')
Merge 08: ('a', 'r')
Merge 09: ('ing', '</w>')
Merge 10: ('a', 't')
Merge 11: ('l', '</w>')
Merge 12: ('l', 'e')
Merge 13: ('r', 'o')
Merge 14: ('d', 'e')
Merge 15: ('r', 'a')
Merge 16: ('c', 'h')
Merge 17: ('er', 's</w>')
Merge 18: ('p', 'ro')
Merge 19: ('en', 'c')
Merge 20: ('w', 'or')
Merge 21: ('le', 'ar')
Merge 22: ('lear', 'n')
Merge 23: ('t', 'o')
Merge 24: ('l', 'a')
Merge 25: ('la', 'ng')
Merge 26: ('lang', 'u')
Merge 27: ('langu', 'a')
Merge 28: ('langua', 'g')
Merge 29: ('languag', 'e</w>')
Merge 30: ('m', 'o')


In [9]:
def encode(sentence, merges):

    sentence = preprocess(sentence)

    encoded_words = []

    for word in sentence.split():

        symbols = list(word)

        symbols.append("</w>")

        for pair in merges:

            i = 0

            while i < len(symbols)-1:

                if (symbols[i], symbols[i+1]) == pair:

                    symbols[i:i+2] = ["".join(pair)]

                else:

                    i += 1

        encoded_words.append(symbols)

    return encoded_words

In [10]:
text = "Transformers improve language"

encoded = encode(text, merges)

print(encoded)

[['t', 'ra', 'n', 's', 'f', 'or', 'm', 'ers</w>'], ['i', 'm', 'pro', 'v', 'e</w>'], ['language</w>']]


In [11]:
def decode(encoded):

    words = []

    for tokens in encoded:

        word = ""

        for token in tokens:

            if token == "</w>":
                continue

            word += token

        words.append(word)

    return " ".join(words)

In [12]:
decoded = decode(encoded)

print("Original :", text)

print("Encoded  :", encoded)

print("Decoded  :", decoded)

Original : Transformers improve language
Encoded  : [['t', 'ra', 'n', 's', 'f', 'or', 'm', 'ers</w>'], ['i', 'm', 'pro', 'v', 'e</w>'], ['language</w>']]
Decoded  : transformers</w> improve</w> language</w>
